In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
"""
Initialize the analysis environment by loading essential modules and setting up
the Python path to access custom analysis functions.
"""

# Enable automatic reloading of modules for interactive development
%load_ext autoreload
%autoreload 2

# Import essential system modules
import sys
from pathlib import Path

# Define the path to custom analysis modules
# Note: Update this path to match your local installation
MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

# Add module path to system path for importing custom functions
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")
print("🔄 Auto-reload enabled for interactive development")

✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis
🔄 Auto-reload enabled for interactive development


In [2]:
from general_utils import find_ephys_sessions
from ephys_dimension_reduction_tdr import plot_tdr_trace_by_quantile

sessions=find_ephys_sessions()

In [5]:
import numpy as np
from ephys_dimension_reduction_tdr import tdr_from_psth
from create_psth import load_zarr
from general_utils import smart_read_csv

failed_sessions = []

for session in sessions[2]:
    try:
        binsize = '0.1'
        time_windows=[[0.3,2],[-1,0]]
  #      latent_vars=['QLearning_L2F1_softmax-choice','QLearning_L2F1_softmax-reward','QLearning_L2F1_softmax-RPE','QLearning_L2F1_softmax-deltaQ','QLearning_L2F1_softmax-sumQ',
  #      'QLearning_L2F1_softmax-chosenQ','QLearning_L2F1_softmax-chosenQ-1','QLearning_L2F1_softmax-unchosenQ-1','QLearning_L2F1_softmax-unchosenQ']
        latent_vars=['QLearning_L1F1_CK1_softmax-sumQ-1','ForagingCompareThreshold-value-1']
  

        for time_window in time_windows:
            for latent_var in latent_vars:
                #time_window=[0.3,2]
                align = "trial_start"
                #latent_var = 'ForagingCompareThreshold-deltaQ-1'
                #latent_var = 'QLearning_L2F1_softmax-deltaQ-1'
                #latent_var = 'QLearning_L2F1_softmax-RPE'
                #latent_var = 'QLearning_L2F1_softmax-choice'
                psth_da = load_zarr(f"/root/capsule/scratch/psth_results/{session}_{binsize}s.zarr")
                df = smart_read_csv(f"/root/capsule/scratch/behavior_summary/behavior_summary-{session}.csv")
                #latent_var = 'QLearning_L2F1_softmax-sumQ-1'
                keep_ids = np.asarray(df['response_trials'][0], dtype=int)
                latent_full = np.asarray(df[latent_var][0], dtype=float)

                out = tdr_from_psth(
                    psth_da,
                    latent=latent_full,
                    align=align,
                    time_window=time_window,
                    include_trials=keep_ids,
                    require_all_ids=True,
                    save_path=f"/root/capsule/scratch/tdr_{session}_{latent_var}_align_{align}_timewindow_{time_window[0]}_{time_window[1]}.zarr",
                    save_format="zarr",
                )

    except Exception as e:
        print(f"❌ Error in session {session}: {e}")
        failed_sessions.append(session)

print("\nAll sessions done.")
if failed_sessions:
    print("Failed sessions:")
    for s in failed_sessions:
        print(f" - {s}")
else:
    print("No errors 🎉")


❌ Error in session ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13: 'latent' length must be either T_full=500 or #included=455.

All sessions done.
Failed sessions:
 - ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13


In [7]:
from behavior_utils import get_fitted_model_names

get_fitted_model_names(session_name='ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13')

No model-fitting results found for subject 844036 on 2026-05-06
